<a href="https://colab.research.google.com/github/rudrarapubhavani/Bhavani/blob/main/Olist_data_read.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import pandas as pd
import os

# Download the dataset
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

# List files
print("Downloaded files:", os.listdir(path))

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Downloaded files: ['olist_customers_dataset.csv', 'olist_sellers_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_order_items_dataset.csv', 'olist_products_dataset.csv', 'olist_geolocation_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_payments_dataset.csv']


In [2]:
# This copies everything from the hidden cache directly to your visible files tab
!cp -r {path}/* /content/

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark= SparkSession.builder.appName('Olive_set').getOrCreate()
path='/content/'


In [8]:
customer_df= spark.read.csv(path+'olist_customers_dataset.csv', header= True, inferSchema= True)
geolocation_df=spark.read.csv(path+'olist_geolocation_dataset.csv', header= True, inferSchema= True)
order_items_df= spark.read.csv(path+'olist_order_items_dataset.csv', header= True, inferSchema= True)
order_payments_df= spark.read.csv(path+'olist_order_payments_dataset.csv', header= True, inferSchema= True)
order_review_df= spark.read.csv(path+'olist_order_reviews_dataset.csv', header= True, inferSchema= True)
orders_df= spark.read.csv(path+'olist_orders_dataset.csv', header= True, inferSchema= True)
product_df= spark.read.csv(path+'olist_products_dataset.csv', header= True, inferSchema= True)
sellers_df= spark.read.csv(path+'olist_sellers_dataset.csv', header=True, inferSchema= True)
product_category= spark.read.csv(path+'product_category_name_translation.csv', header= True, inferSchema= True)




# Data Leakage

In [5]:
print(f'Customer : {customer_df.count()} rows')

Customer : 99441 rows


In [11]:
print(f'Geo Location : {geolocation_df.count()} rows')
print(f'Order Item : {order_items_df.count()} rows')
print(f'Order Payment : {order_payments_df.count()} rows')
print(f'order Review : {order_review_df.count()} rows')


Geo Location : 1000163 rows
Order Item : 112650 rows
Order Payment : 103886 rows
order Review : 104162 rows


In [12]:
print(f'orders : {orders_df.count()} rows')
print(f'Product : {product_df.count()} rows')
print(f'Seller : {sellers_df.count()} rows')
print(f'Product Category : {product_category.count()} rows')

orders : 99441 rows
Product : 32951 rows
Seller : 3095 rows
Product Category : 71 rows


In [17]:
customer_df.createOrReplaceTempView('customer')
geolocation_df.createOrReplaceTempView('geolocation')
order_items_df.createOrReplaceTempView('order_item')
order_payments_df.createOrReplaceTempView('order_payment')
order_review_df.createOrReplaceTempView('order_review')
orders_df.createOrReplaceTempView('order')
product_df.createOrReplaceTempView('product')
sellers_df.createOrReplaceTempView('seller')
product_category.createOrReplaceTempView('product_category')

# Null Values

In [16]:
feilds= ', \n'.join([f'{c} is null as {c}' for c in customer_df.columns])
query= f'select {feilds} from customer'
spark.sql(query).show(5)

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|      false|             false|                   false|        false|         false|
|      false|             false|                   false|        false|         false|
|      false|             false|                   false|        false|         false|
|      false|             false|                   false|        false|         false|
|      false|             false|                   false|        false|         false|
+-----------+------------------+------------------------+-------------+--------------+
only showing top 5 rows


In [20]:
fields= ',\n'.join([f'{c} is null as {c}' for c in geolocation_df.columns ])
query= f'select {fields} from geolocation'
spark.sql(query).show(5)

+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                      false|          false|          false|           false|            false|
|                      false|          false|          false|           false|            false|
|                      false|          false|          false|           false|            false|
|                      false|          false|          false|           false|            false|
|                      false|          false|          false|           false|            false|
+---------------------------+---------------+---------------+----------------+-----------------+
only showing top 5 rows


In [21]:
fields= ','.join([f'{c} is null as {c}' for c in order_items_df.columns])
query= f'select {fields} from order_item'
spark.sql(query).show(5)

+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|   false|        false|     false|    false|              false|false|        false|
|   false|        false|     false|    false|              false|false|        false|
|   false|        false|     false|    false|              false|false|        false|
|   false|        false|     false|    false|              false|false|        false|
|   false|        false|     false|    false|              false|false|        false|
+--------+-------------+----------+---------+-------------------+-----+-------------+
only showing top 5 rows


In [22]:
fields= ','.join([f'{c} is null as {c}' for c in order_payments_df.columns])
query= f'select {fields} from order_payment'
spark.sql(query).show(5)

+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|   false|             false|       false|               false|        false|
|   false|             false|       false|               false|        false|
|   false|             false|       false|               false|        false|
|   false|             false|       false|               false|        false|
|   false|             false|       false|               false|        false|
+--------+------------------+------------+--------------------+-------------+
only showing top 5 rows


In [23]:
fields= ','.join([f'{c} is null as {c}' for c in order_review_df.columns])
query= f'select {fields} from order_review'
spark.sql(query).show(5)

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|    false|   false|       false|                true|                  true|               false|                  false|
|    false|   false|       false|                true|                  true|               false|                  false|
|    false|   false|       false|                true|                  true|               false|                  false|
|    false|   false|       false|                true|                 false|               false|                  false|
|    false|   false|       false|                true|                 false|               false|                  false|
+---------+-----

In [27]:
fields= ','.join([f'sum(case when {c} is null then 1 else 0 end) as {c}' for c in orders_df.columns])
query= f'select {fields} from order'
spark.sql(query).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [28]:
fields= ','.join([f'sum(case when {c} is null then 1 else 0 end) as {c}' for c in product_df.columns])
query= f'select {fields} from product'
spark.sql(query).show()

+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|                610|                       610|               610|               2|                2|                2|               2|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



In [32]:
feilds= ','.join([f'sum(case when {c} is null then 1 else 0 end) as {c} ' for c in sellers_df.columns])
query= f'select {feilds} from seller'
spark.sql(query).show()

+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|        0|                     0|          0|           0|
+---------+----------------------+-----------+------------+



# Duplicate Values


In [36]:
spark.sql(''' select customer_id,count(*) as count from customer group by customer_id having count >1
''').show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [42]:
feilds= ','.join([f'(case when count({c}) > count(distinct({c})) then 1 else 0 end) as {c}' for c in customer_df.columns ])
query= f'select {feilds} from customer'
spark.sql(query).show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 1|                       1|            1|             1|
+-----------+------------------+------------------------+-------------+--------------+



In [45]:
feilds= ','.join([f'count({c}) - count(distinct({c})) as {c}_duplicate_count' for c in customer_df.columns])
query= f'select {feilds} from customer'
spark.sql(query).show()


+---------------------------+----------------------------------+----------------------------------------+-----------------------------+------------------------------+
|customer_id_duplicate_count|customer_unique_id_duplicate_count|customer_zip_code_prefix_duplicate_count|customer_city_duplicate_count|customer_state_duplicate_count|
+---------------------------+----------------------------------+----------------------------------------+-----------------------------+------------------------------+
|                          0|                              3345|                                   84447|                        95322|                         99414|
+---------------------------+----------------------------------+----------------------------------------+-----------------------------+------------------------------+



In [46]:
feilds= ','.join([f'count({c}) - count(distinct({c})) as {c}_duplicate_count' for c in order_items_df.columns])
query= f'select {feilds} from order_item'
spark.sql(query).show()

+------------------------+-----------------------------+--------------------------+-------------------------+-----------------------------------+---------------------+-----------------------------+
|order_id_duplicate_count|order_item_id_duplicate_count|product_id_duplicate_count|seller_id_duplicate_count|shipping_limit_date_duplicate_count|price_duplicate_count|freight_value_duplicate_count|
+------------------------+-----------------------------+--------------------------+-------------------------+-----------------------------------+---------------------+-----------------------------+
|                   13984|                       112629|                     79699|                   109555|                              19332|               106682|                       105651|
+------------------------+-----------------------------+--------------------------+-------------------------+-----------------------------------+---------------------+-----------------------------+



In [48]:
feilds= ','.join([f'count({c}) - count(distinct({c})) as {c}_duplicate_count' for c in order_payments_df.columns])
query= f'select {feilds} from order_payment'
spark.sql(query).show()

+------------------------+----------------------------------+----------------------------+------------------------------------+-----------------------------+
|order_id_duplicate_count|payment_sequential_duplicate_count|payment_type_duplicate_count|payment_installments_duplicate_count|payment_value_duplicate_count|
+------------------------+----------------------------------+----------------------------+------------------------------------+-----------------------------+
|                    4446|                            103857|                      103881|                              103862|                        74809|
+------------------------+----------------------------------+----------------------------+------------------------------------+-----------------------------+



In [49]:
feilds= ','.join([f'count({c}) - count(distinct({c})) as {c}_duplicate_count' for c in order_review_df.columns])
query= f'select {feilds} from order_review'
spark.sql(query).show()

+-------------------------+------------------------+----------------------------+------------------------------------+--------------------------------------+------------------------------------+---------------------------------------+
|review_id_duplicate_count|order_id_duplicate_count|review_score_duplicate_count|review_comment_title_duplicate_count|review_comment_message_duplicate_count|review_creation_date_duplicate_count|review_answer_timestamp_duplicate_count|
+-------------------------+------------------------+----------------------------+------------------------------------+--------------------------------------+------------------------------------+---------------------------------------+
|                     1204|                    2184|                       99339|                                7058|                                  5177|                               94677|                                    919|
+-------------------------+------------------------+--------

In [50]:
feilds= ','.join([f'count({c}) - count(distinct({c})) as {c}_duplicate_count' for c in orders_df.columns])
query= f'select {feilds} from order'
spark.sql(query).show()

+------------------------+---------------------------+----------------------------+----------------------------------------+---------------------------------+--------------------------------------------+---------------------------------------------+---------------------------------------------+
|order_id_duplicate_count|customer_id_duplicate_count|order_status_duplicate_count|order_purchase_timestamp_duplicate_count|order_approved_at_duplicate_count|order_delivered_carrier_date_duplicate_count|order_delivered_customer_date_duplicate_count|order_estimated_delivery_date_duplicate_count|
+------------------------+---------------------------+----------------------------+----------------------------------------+---------------------------------+--------------------------------------------+---------------------------------------------+---------------------------------------------+
|                       0|                          0|                       99433|                             

In [51]:
feilds= ','.join([f'count({c}) - count(distinct({c})) as {c}_duplicate_count' for c in product_df.columns])
query= f'select {feilds} from product'
spark.sql(query).show()

+--------------------------+-------------------------------------+-----------------------------------+------------------------------------------+----------------------------------+--------------------------------+---------------------------------+---------------------------------+--------------------------------+
|product_id_duplicate_count|product_category_name_duplicate_count|product_name_lenght_duplicate_count|product_description_lenght_duplicate_count|product_photos_qty_duplicate_count|product_weight_g_duplicate_count|product_length_cm_duplicate_count|product_height_cm_duplicate_count|product_width_cm_duplicate_count|
+--------------------------+-------------------------------------+-----------------------------------+------------------------------------------+----------------------------------+--------------------------------+---------------------------------+---------------------------------+--------------------------------+
|                         0|                           

In [52]:
feilds= ','.join([f'count({c}) - count(distinct({c})) as {c}_duplicate_count' for c in sellers_df.columns])
query= f'select {feilds} from seller'
spark.sql(query).show()

+-------------------------+--------------------------------------+---------------------------+----------------------------+
|seller_id_duplicate_count|seller_zip_code_prefix_duplicate_count|seller_city_duplicate_count|seller_state_duplicate_count|
+-------------------------+--------------------------------------+---------------------------+----------------------------+
|                        0|                                   849|                       2484|                        3072|
+-------------------------+--------------------------------------+---------------------------+----------------------------+

